# MAICON 모의고사 1회차 · 문제 3 — 장비 고장 여부 예측 (AI4I 2020)

- 목표: 고장(1)을 양성 클래스로 하는 **F1-score** 최대화, 제출 `id,machine_failure`
- 환경: Python 3.9 / pandas 1.5 / scikit-learn 1.1 호환, **sklearn만 사용**(추가 설치 없음), CPU 2코어
- 접근 요약
  1. RandomForest 기본 모델(원본 6개 입력) → 베이스라인
  2. 물리적으로 의미 있는 파생 변수(온도차, 기계적 출력, 마모×토크의 형식별 한계 대비 비율)와 고장 조건 플래그 추가
  3. 반복 층화 교차검증(5겹 × 3회)으로 같은 분할에서 비교, 오류 분석으로 천장 확인
  4. 분포 이동 점검 후 최종 제출

In [1]:
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings('ignore')
T0 = time.time()

# 대회 서버/로컬 어디서든 돌도록 경로 후보를 여러 개 둔다.
CANDIDATES = [Path('data'), Path('.'), Path('/workspace/problem03/data'),
              Path('/home/jovyan/work/Problem #3/data'), Path('/home/jovyan/work/Problem #3')]
DATA_DIR = next(p for p in CANDIDATES if (p / 'train.csv').exists())
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('DATA_DIR:', DATA_DIR)
print('train/test/sample:', train.shape, test.shape, sample.shape)
print('결측치:', int(train.isna().sum().sum()), int(test.isna().sum().sum()))
print('고장 비율(train): %.4f (%d / %d)' % (train.machine_failure.mean(), train.machine_failure.sum(), len(train)))
print('형식별 고장 비율:'); print(train.groupby('type').machine_failure.agg(['count', 'sum', 'mean']).round(4))
assert set(test.id) == set(sample.id) and not set(train.id) & set(test.id)

DATA_DIR: data
train/test/sample: (8000, 8) (2000, 7) (2000, 2)
결측치: 0 0
고장 비율(train): 0.0339 (271 / 8000)
형식별 고장 비율:
      count  sum    mean
type                    
H       814   19  0.0233
L      4776  181  0.0379
M      2410   71  0.0295


## 1. 고장 메커니즘 가설 확인 (train만 사용)

AI4I 2020 원본 설명에는 고장 유형이 센서값의 규칙으로 정의되어 있다(공개 문서 내용, 문제 설명의 원본 데이터 링크).
이 규칙들을 **정답 열 없이 센서값만으로 계산한 특징**으로 만들고, train에서 실제로 맞는지 먼저 확인한다.

| 유형 | 조건 |
|---|---|
| HDF 방열 | 공정-공기 온도차 < 8.6 K 그리고 회전 속도 < 1380 rpm |
| PWF 출력 | 토크 × 각속도(W) < 3500 또는 > 9000 |
| OSF 과부하 | 마모 × 토크 > 형식별 한계 (L 11000, M 12000, H 13000) |
| TWF 마모 | 마모 200~240분 사이 무작위 교체/고장 (센서로 예측 불가) |
| RNF | 0.1% 무작위 (예측 불가) |

In [2]:
THRESH = {'L': 11000.0, 'M': 12000.0, 'H': 13000.0}

def add_features(d):
    x = pd.DataFrame(index=d.index)
    x['air'] = d.air_temperature_k
    x['proc'] = d.process_temperature_k
    x['rpm'] = d.rotational_speed_rpm
    x['torque'] = d.torque_nm
    x['wear'] = d.tool_wear_min
    x['type'] = d.type.map({'L': 0, 'M': 1, 'H': 2})
    # 파생 변수. 온도차는 반올림하지 않고 float64 그대로 쓴다(경계값 8.6 근처에서 중요, 아래 참고).
    x['diff'] = d.process_temperature_k - d.air_temperature_k
    x['power'] = d.torque_nm * d.rotational_speed_rpm * 2 * np.pi / 60
    x['strain'] = d.tool_wear_min * d.torque_nm
    x['strain_ratio'] = x.strain / d.type.map(THRESH)
    # 규칙 플래그 (센서값만으로 계산)
    x['osf'] = (x.strain_ratio > 1).astype(int)
    x['hdf'] = ((x['diff'] < 8.6) & (d.rotational_speed_rpm < 1380)).astype(int)
    x['pwf'] = ((x.power < 3500) | (x.power > 9000)).astype(int)
    x['rule'] = ((x.osf + x.hdf + x.pwf) > 0).astype(int)
    return x

Xtr, Xte = add_features(train), add_features(test)
y = train.machine_failure.values
check = pd.DataFrame({f: {'플래그 수': int(Xtr[f].sum()),
                          '그중 고장': int(y[Xtr[f] == 1].sum()),
                          '고장 비율': round(float(y[Xtr[f] == 1].mean()), 4)} for f in ['osf', 'hdf', 'pwf', 'rule']}).T
print(check)
print('규칙에 안 걸린 고장:', int(y[Xtr.rule == 0].sum()), '/ 전체 고장', int(y.sum()))

      플래그 수  그중 고장  고장 비율
osf    76.0   76.0    1.0
hdf    94.0   94.0    1.0
pwf    76.0   76.0    1.0
rule  229.0  229.0    1.0
규칙에 안 걸린 고장: 42 / 전체 고장 271


**경계값 주의(HDF)**: 온도는 소수 첫째 자리 값이라 `process - air`가 8.6인 행이 많다. 그런데 float64 뺄셈은 어떤 쌍에서는 8.5999…, 어떤 쌍에서는 8.6000…이 된다.
원본 라벨이 이 float 결과로 만들어졌는지 해당 행들로 직접 확인한다. (반올림해서 `<= 8.6`으로 쓰면 오탐이 생기는지도 같이 본다.)

In [3]:
d_float = train.process_temperature_k - train.air_temperature_k
band = (d_float.round(1) == 8.6) & (train.rotational_speed_rpm < 1380)
tab = pd.crosstab(pd.Series(np.where(d_float[band] < 8.6, 'float diff < 8.6', 'float diff >= 8.6'), index=d_float[band].index),
                  train.machine_failure[band], rownames=['8.6 근처 행'], colnames=['실제 고장'])
print(tab)
for name, hd in [('float 그대로 < 8.6', d_float < 8.6), ('반올림 < 8.6', d_float.round(1) < 8.6), ('반올림 <= 8.6', d_float.round(1) <= 8.6)]:
    flag = hd & (train.rotational_speed_rpm < 1380)
    print('%-16s 플래그 %3d  그중 고장 %3d  오탐 %2d' % (name, flag.sum(), y[flag].sum(), (flag & (train.machine_failure == 0)).sum()))

실제 고장               0   1
8.6 근처 행                 
float diff < 8.6    0  11
float diff >= 8.6  11   2
float 그대로 < 8.6  플래그  94  그중 고장  94  오탐  0
반올림 < 8.6        플래그  83  그중 고장  83  오탐  0
반올림 <= 8.6       플래그 107  그중 고장  96  오탐 11


## 2. 교차검증 비교 (같은 5겹 × 3회 반복 분할, 임계값 0.5 고정)

비교 대상
- `RF_raw` : 원본 6개 입력만 쓰는 RandomForest(베이스라인)
- `RF_eng` : 파생 변수(온도차·출력·마모×토크 비율)만 추가
- `RF_eng+flags` : 규칙 플래그까지 추가 (**메인 후보**)
- `ET_eng+flags` : ExtraTrees 대체
- `rule_only` : 학습 없이 규칙 플래그만 사용(모델 없는 기준선)

In [4]:
RAW = ['air', 'proc', 'rpm', 'torque', 'wear', 'type']
ENG = ['diff', 'rpm', 'power', 'strain_ratio', 'wear', 'type']
FLG = ENG + ['osf', 'hdf', 'pwf']
def rf(**kw):
    return RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42, **kw)
MODELS = {
    'RF_raw':       (RAW, lambda: rf()),
    'RF_eng':       (ENG, lambda: rf()),
    'RF_eng+flags': (FLG, lambda: rf(min_samples_leaf=3)),
    'ET_eng+flags': (FLG, lambda: ExtraTreesClassifier(n_estimators=300, n_jobs=2, random_state=42)),
    'rule_only':    (None, None),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
splits = list(cv.split(Xtr, y))
oof = {k: np.zeros((3, len(y))) for k in MODELS}        # 반복 회차별 OOF 확률(또는 라벨)
fold_f1 = {k: [] for k in MODELS}
t1 = time.time()
for i, (a, b) in enumerate(splits):
    rep = i // 5
    for k, (cols, mk) in MODELS.items():
        if mk is None:
            p = Xtr.rule.values[b].astype(float)
        else:
            m = mk().fit(Xtr.iloc[a][cols], y[a])
            p = m.predict_proba(Xtr.iloc[b][cols])[:, 1]
        oof[k][rep, b] = p
        fold_f1[k].append(f1_score(y[b], p >= 0.5))
print('CV 소요 %.0f초' % (time.time() - t1))
rows = []
for k in MODELS:
    pooled = [f1_score(y, oof[k][r] >= 0.5) for r in range(3)]
    lab = oof[k] >= 0.5
    rows.append({'model': k, 'F1(3회 평균)': np.mean(pooled), 'F1 회차별': np.round(pooled, 4),
                 'precision': np.mean([precision_score(y, lab[r]) for r in range(3)]),
                 'recall': np.mean([recall_score(y, lab[r]) for r in range(3)]),
                 'fold F1 표준편차': np.std(fold_f1[k])})
cv_table = pd.DataFrame(rows).set_index('model')
print(cv_table.round(4).to_string())

CV 소요 40초
              F1(3회 평균)                    F1 회차별  precision  recall  fold F1 표준편차
model                                                                             
RF_raw           0.6984  [0.7105, 0.6815, 0.7031]     0.8654  0.5855        0.0579
RF_eng           0.8813  [0.8773, 0.8813, 0.8853]     0.9690  0.8081        0.0248
RF_eng+flags     0.9142  [0.9142, 0.9142, 0.9142]     0.9957  0.8450        0.0178
ET_eng+flags     0.9136   [0.9124, 0.9124, 0.916]     0.9942  0.8450        0.0178
rule_only        0.9160     [0.916, 0.916, 0.916]     1.0000  0.8450        0.0189


## 3. 오류 분석 — 남은 오류가 줄일 수 있는 오류인가?

규칙에 걸리지 않은 고장(= 모델이 놓치는 고장)이 마모 구간별로 얼마나 되는지 본다.
구간 내 고장 확률이 약 0.46(= F1/2) 아래면 양성으로 예측하는 것이 F1에 손해다.

In [5]:
res = Xtr[Xtr.rule == 0].copy(); res['y'] = y[Xtr.rule == 0]
bins = pd.cut(res.wear, [-1, 100, 180, 197, 215, 235, 251, 300])
tbl = res.groupby(bins).y.agg(['count', 'sum', 'mean']).rename(columns={'count': '행', 'sum': '고장', 'mean': '고장 비율'})
print(tbl.round(4).to_string())
n_miss = int(res.y.sum())
print('\n규칙 밖 고장 %d건 = 마모 195~251 구간(TWF 추정) %d건 + 그 밖(RNF 추정) %d건' % (
    n_miss, int(res[(res.wear >= 195) & (res.wear <= 251)].y.sum()), int(res[(res.wear < 195) | (res.wear > 251)].y.sum())))
best = tbl['고장 비율'].max()
print('규칙 밖에서 가장 높은 구간 고장 비율 %.3f  (양성 예측 이득 기준 약 0.46)' % best)
ceiling = 2 * (int(y.sum()) - n_miss) / (2 * (int(y.sum()) - n_miss) + n_miss)
print('센서 규칙으로 맞힐 수 있는 상한(규칙 밖 고장을 못 잡는다고 볼 때) F1 = %.4f' % ceiling)

               행  고장   고장 비율
wear                        
(-1, 100]   3638   2  0.0005
(100, 180]  2887   4  0.0014
(180, 197]   599   0  0.0000
(197, 215]   505  19  0.0376
(215, 235]   128  16  0.1250
(235, 251]    14   1  0.0714

규칙 밖 고장 42건 = 마모 195~251 구간(TWF 추정) 36건 + 그 밖(RNF 추정) 6건
규칙 밖에서 가장 높은 구간 고장 비율 0.125  (양성 예측 이득 기준 약 0.46)
센서 규칙으로 맞힐 수 있는 상한(규칙 밖 고장을 못 잡는다고 볼 때) F1 = 0.9160


## 4. 최종 모델 학습과 분포 이동 점검

메인: `RF_eng+flags`(규칙 플래그 포함 RandomForest, 전체 train으로 재학습). 비교용으로 `rule_only`와 `RF_raw` 예측도 만든다.

In [6]:
cols_main, mk_main = MODELS['RF_eng+flags']
main = mk_main().fit(Xtr[cols_main], y)
prob_test = main.predict_proba(Xte[cols_main])[:, 1]
pred_main = (prob_test >= 0.5).astype(int)
pred_rule = Xte.rule.values.astype(int)
base = MODELS['RF_raw'][1]().fit(Xtr[RAW], y)
pred_base = (base.predict_proba(Xte[RAW])[:, 1] >= 0.5).astype(int)

shift = pd.DataFrame({
    'train': [Xtr.rule.mean(), Xtr.osf.mean(), Xtr.hdf.mean(), Xtr.pwf.mean(), y.mean(), Xtr.wear.mean(), Xtr.torque.mean(), Xtr.rpm.mean()],
    'test':  [Xte.rule.mean(), Xte.osf.mean(), Xte.hdf.mean(), Xte.pwf.mean(), pred_main.mean(), Xte.wear.mean(), Xte.torque.mean(), Xte.rpm.mean()]},
    index=['규칙 플래그 비율', 'OSF', 'HDF', 'PWF', '고장 비율(train 실제 / test 예측)', '마모 평균', '토크 평균', 'rpm 평균']).round(4)
print(shift.to_string())
print('\n형식 비율  train', train.type.value_counts(normalize=True).round(3).to_dict(), ' test', test.type.value_counts(normalize=True).round(3).to_dict())
print('test 예측 양성 수: 메인 %d / 규칙만 %d / RF_raw %d' % (pred_main.sum(), pred_rule.sum(), pred_base.sum()))
print('메인 vs 규칙만 불일치 %d건, 메인 vs RF_raw 불일치 %d건' % ((pred_main != pred_rule).sum(), (pred_main != pred_base).sum()))

                               train       test
규칙 플래그 비율                     0.0286     0.0290
OSF                           0.0095     0.0110
HDF                           0.0118     0.0105
PWF                           0.0095     0.0095
고장 비율(train 실제 / test 예측)     0.0339     0.0290
마모 평균                       107.9058   108.1320
토크 평균                        39.9267    40.2279
rpm 평균                     1539.4034  1536.2670

형식 비율  train {'L': 0.597, 'M': 0.301, 'H': 0.102}  test {'L': 0.612, 'M': 0.294, 'H': 0.094}
test 예측 양성 수: 메인 58 / 규칙만 58 / RF_raw 54
메인 vs 규칙만 불일치 0건, 메인 vs RF_raw 불일치 22건


## 5. 제출 파일 저장과 형식 검증

In [7]:
sub = pd.DataFrame({'id': test.id.values, 'machine_failure': pred_main.astype(int)})
sub = sample[['id']].merge(sub, on='id', how='left', validate='one_to_one')
sub.to_csv('submission.csv', index=False)

# 버전 보관(같은 형식)
vdir = Path('versions'); vdir.mkdir(exist_ok=True)
for name, pred in [('v1_rf_raw_baseline', pred_base), ('v3_rule_only', pred_rule), ('v4_rf_eng_flags_main', pred_main)]:
    pd.DataFrame({'id': sample.id, 'machine_failure': pred.astype(int)}).to_csv(vdir / (name + '.csv'), index=False)

chk = pd.read_csv('submission.csv')
assert chk.columns.tolist() == ['id', 'machine_failure']
assert chk.id.tolist() == sample.id.tolist()
assert chk.id.is_unique and not chk.isna().any().any()
assert chk.machine_failure.isin([0, 1]).all() and chk.machine_failure.dtype.kind == 'i'
assert len(chk) == len(sample) == 2000
print('제출 형식 검증 통과:', chk.shape, chk.machine_failure.value_counts().to_dict())
print('총 소요 %.0f초' % (time.time() - T0))

제출 형식 검증 통과: (2000, 2) {0: 1942, 1: 58}
총 소요 42초
